# Example analysis

The pattern every notebook in this repository follows. Copy it, replace the content, keep the shape.

**Four rules, all enforced by `tests/test_template_compliance.py`:**

1. **No `def`, no `class`.** All logic lives in `src/` and is imported here. A function defined in a notebook cannot be imported, cannot be tested, and gets copied into the next notebook — where the two copies then diverge.
2. **Figures are saved by this notebook**, through `FigureSaver`, into `output/figures/example_analysis/`, as PDF *and* PNG. The setup cell clears that folder first, so a stale figure can never survive next to a fresh one.
3. **Never pick a colour here.** `src/visualize/style.py` owns every colour, and a name means the same colour in every figure in the project.
4. **The last code cell prints a text summary** of everything shown. `output/All_Results.md` is built from what the notebooks print.

In [ ]:
# Setup. Run this first, always.
import sys
from pathlib import Path

# The repository root on sys.path. WHY: Jupyter's working directory is notebooks/, so
# `from src...` does not resolve without this. The notebook runner sets cwd to the root
# instead, which is why this line is harmless there — and why it must be here, so an
# interactive run behaves identically.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.visualize import figures, style

style.apply()

# Declare this notebook's series ONCE, in slot order. Append, never insert: inserting a
# name in the middle repaints every figure after it and silently invalidates the ones
# already in the paper. For a project-wide legend, move this call into style.py.
style.register_series("variant A", "variant B", "variant C")

# Constructing the saver CLEARS this notebook's own figure folder — before anything is
# drawn, and never another notebook's.
NOTEBOOK = "example_analysis"
save = figures.FigureSaver(NOTEBOOK)

print(f"figures -> {save.folder}")

## Data

Replace this cell with a call into `src/data/`. The synthetic draw below exists only so the template runs on a fresh clone, before any real data is in `data/raw/`.

In [ ]:
# PLACEHOLDER. Real notebooks call src.data.loaders.load(...) instead.
rng = np.random.default_rng(20260811)
N = 2000

measurements = {
    "variant A": rng.normal(0.72, 0.05, N),
    "variant B": rng.normal(0.76, 0.04, N),
    "variant C": rng.normal(0.69, 0.07, N),
}
baseline_score = 0.70

print(f"{len(measurements)} variants, {N} draws each")

In [ ]:
# Figure 1 — distributions, one colour per variant, from the shared palette.
fig, ax = plt.subplots(figsize=style.figsize(style.WIDTH_DOUBLE, 0.42))

for label, values in measurements.items():
    ax.hist(values, bins=40, histtype="step", linewidth=1.8,
            color=style.color(label), label=label)

# A reference line is always the annotation colour and always dashed, in every figure, so
# it can never be mistaken for data.
style.annotate_reference(ax, baseline_score, "baseline", axis="x")
ax.set_xlabel("score")
ax.set_ylabel("count")
ax.set_title("Score distribution per variant")
style.legend(ax)

save(fig, "score_distributions", caption=(
    "Step histograms of the score for three variants, 40 bins each, 2,000 draws per "
    "variant. The dashed vertical line marks the baseline score of 0.70."
))

In [ ]:
# Figure 2 — means with a spread, ordered. Bars, so the full eight-slot palette applies.
labels = sorted(measurements, key=lambda k: measurements[k].mean(), reverse=True)
means = [measurements[k].mean() for k in labels]
spread = [measurements[k].std() for k in labels]

fig, ax = plt.subplots(figsize=style.figsize(style.WIDTH_SINGLE, 0.75))
ax.bar(labels, means, yerr=spread, color=style.series_colors(labels), width=0.62)
style.annotate_reference(ax, baseline_score, "baseline")
ax.set_ylabel("mean score")
ax.set_ylim(0.5, 0.9)
ax.set_title("Mean score, ordered")

# Direct labels, not just colour: three of the eight palette slots sit below 3:1 contrast
# against white, so a reader must never have to identify a bar by its colour alone.
# Offset above the ERROR BAR, not above the bar top — otherwise the label sits inside the
# whisker and reads as part of it.
for x, (value, sd) in enumerate(zip(means, spread)):
    ax.annotate(f"{value:.3f}", (x, value + sd), textcoords="offset points", xytext=(0, 5),
                ha="center", fontsize=7, color=style.INK["secondary"])

save(fig, "mean_score_ranking", caption=(
    "Mean score per variant, ordered by mean, with error bars giving one standard "
    "deviation across 2,000 draws. The dashed horizontal line marks the baseline score "
    "of 0.70. Values are printed above each bar."
))

## Summary

The last code cell. Everything it prints ends up in `output/All_Results.md`, so print the numbers a reader would otherwise have to squint at a figure to recover.

In [ ]:
print("=" * 70)
print(f"{NOTEBOOK}: score comparison across {len(measurements)} variants")
print("=" * 70)
print(f"{N} draws per variant, baseline = {baseline_score:.3f}")
print()
print(f"{'variant':<12} {'mean':>8} {'std':>8} {'vs baseline':>12}")
for label in labels:
    values = measurements[label]
    print(f"{label:<12} {values.mean():>8.4f} {values.std():>8.4f} "
          f"{values.mean() - baseline_score:>+12.4f}")
print()
print(f"best: {labels[0]} at {measurements[labels[0]].mean():.4f}")
print()
print(save.summary())